# BlueShift: Aggregate NASA Provincial Quarterly Data

We will use the existing `mindanao_coastal_ocean_zones.gpkg`, where each province has its own exclusive offshore zone. We'll calculate statistics from the NASA raster pixels inside each province's zone, then aggregate the monthly results into quarters.

**Aggregation approach**
1. **Province-level monthly statistics:** Calculate the spatial mean of chlorophyll-a and SST for each province and month.
2. **Quarterly aggregation:** Calculate the average of the available monthly provincial means for each quarter. This gives each month equal weight rather than allowing months with more valid pixels to dominate.
3. **One final CSV:** One row per province per quarter, with both ocean variables and data-availability indicators.

We'll retain incomplete quarters but flag them. For example, Q2 2026 will be marked incomplete because June is unavailable, and chlorophyll-a is also missing for May.

**Expected output**

| Column                 | Description                               |
| ---------------------- | ----------------------------------------- |
| `period_type`          | monthly or quarterly                      |
| `period`               | e.g. 2002-07 or 2002-Q3                   |
| `province_id`          | Province identifier                       |
| `province`             | Province name                             |
| `chlor_a_mean`         | Mean chlorophyll-a (mg/m³)                |
| `sst_mean`             | Mean SST (°C)                             |
| `chlor_a_months`       | Months contributing to the quarterly mean |
| `sst_months`           | Months contributing to the quarterly mean |
| `chlor_a_valid_pixels` | Valid pixel observations                  |
| `sst_valid_pixels`     | Valid pixel observations                  |

## 1. Imports and paths

In [1]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import regionmask
import xarray as xr

BASE_DIR = Path("..")

ZONES_FILE = BASE_DIR / "data/processed/ocean_zones/mindanao_coastal_ocean_zones.gpkg"

CHL_DIR = BASE_DIR / "data/processed/nasa/chlorophyll_a"
SST_DIR = BASE_DIR / "data/processed/nasa/sst"

OUTPUT_DIR = BASE_DIR / "data/processed/nasa"
OUTPUT_FILE = OUTPUT_DIR / "nasa_provincial_quarterly.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

D:\miniforge3\envs\blueshift\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load provincial coastal zones

In [2]:
zones = gpd.read_file(ZONES_FILE).to_crs("EPSG:4326")

zones = zones.sort_values("province_id").reset_index(drop=True)

zones[["province_id", "province"]].head()

,province_id,province
0,1,Zamboanga del Norte
1,2,Zamboanga del Sur
2,3,Zamboanga Sibugay
3,4,Camiguin
4,5,Lanao del Norte


## 3. Check downloaded monthly files

In [3]:
chl_files = sorted(CHL_DIR.glob("chlorophyll_a_*.nc"))
sst_files = sorted(SST_DIR.glob("sst_*.nc"))

def get_months(files):
    return sorted(
        file.stem.split("_")[-1]
        for file in files
    )

chl_months = get_months(chl_files)
sst_months = get_months(sst_files)

print("Chlorophyll-a files:", len(chl_files))
print("First:", chl_months[0] if chl_months else "None")
print("Last:", chl_months[-1] if chl_months else "None")

print("\nSST files:", len(sst_files))
print("First:", sst_months[0] if sst_months else "None")
print("Last:", sst_months[-1] if sst_months else "None")

Chlorophyll-a files: 286
First: 200207
Last: 202604

SST files: 287
First: 200207
Last: 202605


## 4. Create an aggregation function

This function calculates the mean and valid pixel count for each province.

In [4]:
# Assign a unique numeric ID to each province
zones = zones.reset_index(drop=True).copy()
zones["region_number"] = np.arange(len(zones))
region_numbers = zones["region_number"].to_numpy()

def aggregate_file(file_path, variable):

    if not file_path.exists():
        return (
            [np.nan] * len(zones),
            [0] * len(zones),
            [0] * len(zones)
        )

    with xr.open_dataset(file_path) as ds:

        data = ds[variable]

        # Create mask using this file's exact coordinates
        mask = regionmask.mask_geopandas(
            zones,
            numbers="region_number",
            lon_or_obj=ds["lon"],
            lat=ds["lat"]
        )

        if variable == "chlor_a":
            data = data.where((data >= 0.001) & (data <= 100))

        elif variable == "sst":
            data = data.where(ds["qual_sst"].isin([0, 1]))

        else:
            raise ValueError(f"Unsupported variable: {variable}")

        means = []
        counts = []
        mask_counts = []

        for region_number in region_numbers:

            province_mask = mask == region_number
            province_data = data.where(province_mask)

            mask_count = int(province_mask.sum().item())
            count = int(province_data.count().item())

            mean = (
                float(province_data.mean(skipna=True).item())
                if count > 0 else np.nan
            )

            means.append(mean)
            counts.append(count)
            mask_counts.append(mask_count)

    return means, counts, mask_counts

The quality filters are repeated here as a safeguard, even though they were already applied during extraction.

## 5. Aggregate monthly data by province

We'll process July 2002 through May 2026. Each month's CHL and SST files will use their own spatial masks.

In [5]:

months = pd.period_range(
    start="2002-07",
    end="2026-05",
    freq="M"
)

monthly_rows = []

for month in months:

    month_id = month.strftime("%Y%m")

    chl_means, chl_counts, chl_mask_counts = aggregate_file(
        CHL_DIR / f"chlorophyll_a_{month_id}.nc",
        "chlor_a"
    )

    sst_means, sst_counts, sst_mask_counts = aggregate_file(
        SST_DIR / f"sst_{month_id}.nc",
        "sst"
    )

    for i, zone in zones.iterrows():

        monthly_rows.append({
            "period_type": "monthly",
            "period": str(month),
            "year": month.year,
            "quarter": f"Q{month.quarter}",
            "month": month.month,
            "province_id": zone["province_id"],
            "province": zone["province"],

            "chlor_a_mean": chl_means[i],
            "chlor_a_valid_pixels": chl_counts[i],
            "chlor_a_mask_pixels": chl_mask_counts[i],

            "sst_mean": sst_means[i],
            "sst_valid_pixels": sst_counts[i],
            "sst_mask_pixels": sst_mask_counts[i]
        })

monthly_df = pd.DataFrame(monthly_rows)

monthly_df.head()

,period_type,period,year,quarter,month,province_id,province,chlor_a_mean,chlor_a_valid_pixels,chlor_a_mask_pixels,sst_mean,sst_valid_pixels,sst_mask_pixels
0,monthly,2002-07,2002,Q3,7,1,Zamboanga del Norte,0.215215,271,274,29.596478,274,274
1,monthly,2002-07,2002,Q3,7,2,Zamboanga del Sur,0.587525,217,225,29.569334,225,225
2,monthly,2002-07,2002,Q3,7,3,Zamboanga Sibugay,0.616096,117,123,29.568089,123,123
3,monthly,2002-07,2002,Q3,7,4,Camiguin,0.192711,64,64,29.689529,64,64
4,monthly,2002-07,2002,Q3,7,5,Lanao del Norte,0.620352,43,52,29.968365,52,52


## 6. Assess coverage across all provinces

This calculates monthly coverage using the actual number of pixels assigned to each province in each file.

In [6]:
# Calculate monthly coverage percentages
coverage_df = monthly_df.copy()

coverage_df["sst_coverage_pct"] = (
    coverage_df["sst_valid_pixels"]
    / coverage_df["sst_mask_pixels"].replace(0, np.nan)
    * 100
)

coverage_df["chl_coverage_pct"] = (
    coverage_df["chlor_a_valid_pixels"]
    / coverage_df["chlor_a_mask_pixels"].replace(0, np.nan)
    * 100
)

# Summarize by province
province_coverage = (
    coverage_df.groupby(
        ["province_id", "province"],
        as_index=False
    )
    .agg(
        sst_mean_coverage=("sst_coverage_pct", "mean"),
        sst_zero_coverage_months=(
            "sst_valid_pixels",
            lambda x: (x == 0).sum()
        ),
        chl_mean_coverage=("chl_coverage_pct", "mean"),
        chl_zero_coverage_months=(
            "chlor_a_valid_pixels",
            lambda x: (x == 0).sum()
        )
    )
)

province_coverage.round(2).sort_values(
    "province_id"
)

,province_id,province,sst_mean_coverage,sst_zero_coverage_months,chl_mean_coverage,chl_zero_coverage_months
0,1,Zamboanga del Norte,99.56,0,95.83,1
1,2,Zamboanga del Sur,99.80,0,96.64,1
2,3,Zamboanga Sibugay,99.98,0,97.04,1
3,4,Camiguin,99.79,0,99.14,1
4,5,Lanao del Norte,99.56,0,95.82,1
5,6,Misamis Occidental,99.40,0,92.32,1
6,7,Misamis Oriental,99.46,0,96.88,1
7,8,Davao del Norte,99.75,0,97.03,1
8,9,Davao del Sur,99.53,0,94.40,1
9,10,Davao Oriental,99.75,0,97.89,1


This summary tells us the average monthly spatial coverage and how many months had zero valid observations for each province.

## 7. Imputation

No imputation for now.

We'll retain missing monthly means as NaN. This preserves the distinction between actual satellite observations and missing observations.

Before proceeding, inspect the remaining missing values:

In [7]:
print("Monthly records:", len(monthly_df))

print("\nMissing monthly means:")
print(
    monthly_df[
        ["chlor_a_mean", "sst_mean"]
    ].isna().sum()
)

print("\nZero valid observation counts:")
print(
    "CHL:", (monthly_df["chlor_a_valid_pixels"] == 0).sum()
)
print(
    "SST:", (monthly_df["sst_valid_pixels"] == 0).sum()
)

Monthly records: 6888

Missing monthly means:
chlor_a_mean    41
sst_mean         0
dtype: int64

Zero valid observation counts:
CHL: 41
SST: 0


## 8. Aggregate quarterly data

Quarterly means will be calculated using available monthly provincial means. Pandas automatically skips NaN values.

Each available month receives equal weight, regardless of its valid pixel count.

In [8]:
quarterly_df = (
    monthly_df
    .groupby(
        ["province_id", "province", "year", "quarter"],
        as_index=False
    )
    .agg(
        # Quarterly means from available monthly observations
        chlor_a_mean=("chlor_a_mean", "mean"),
        sst_mean=("sst_mean", "mean"),

        # Number of months with actual observations
        chlor_a_months_observed=("chlor_a_mean", "count"),
        sst_months_observed=("sst_mean", "count"),

        # Actual satellite pixel counts
        chlor_a_valid_pixels=("chlor_a_valid_pixels", "sum"),
        sst_valid_pixels=("sst_valid_pixels", "sum"),

        # Total pixels assigned to each provincial zone
        chlor_a_mask_pixels=("chlor_a_mask_pixels", "sum"),
        sst_mask_pixels=("sst_mask_pixels", "sum")
    )
)

quarterly_df["period_type"] = "quarterly"

quarterly_df["period"] = (
    quarterly_df["year"].astype(str)
    + "-"
    + quarterly_df["quarter"].astype(str)
)

quarterly_df.head()

,province_id,province,year,quarter,chlor_a_mean,sst_mean,chlor_a_months_observed,sst_months_observed,chlor_a_valid_pixels,sst_valid_pixels,chlor_a_mask_pixels,sst_mask_pixels,period_type,period
0,1,Zamboanga del Norte,2002,Q3,0.246262,29.405954,3,3,743,821,822,822,quarterly,2002-Q3
1,1,Zamboanga del Norte,2002,Q4,0.872823,29.253698,3,3,820,822,822,822,quarterly,2002-Q4
2,1,Zamboanga del Norte,2003,Q1,1.704142,26.430674,3,3,820,822,822,822,quarterly,2003-Q1
3,1,Zamboanga del Norte,2003,Q2,0.802028,29.159945,3,3,816,822,822,822,quarterly,2003-Q2
4,1,Zamboanga del Norte,2003,Q3,0.330493,29.832467,3,3,755,796,822,822,quarterly,2003-Q3


## 9. Check quarterly completeness

This helps identify quarters with incomplete observations.

In [9]:
print("Quarterly records:", len(quarterly_df))

print("\nCHL months observed:")
print(quarterly_df["chlor_a_months_observed"].value_counts().sort_index())

print("\nSST months observed:")
print(quarterly_df["sst_months_observed"].value_counts().sort_index())

print("\nQuarters with no observations:")
print(
    "CHL:", (quarterly_df["chlor_a_months_observed"] == 0).sum()
)
print(
    "SST:", (quarterly_df["sst_months_observed"] == 0).sum()
)

Quarterly records: 2304

CHL months observed:
chlor_a_months_observed
1      24
2      17
3    2263
Name: count, dtype: int64

SST months observed:
sst_months_observed
2      24
3    2280
Name: count, dtype: int64

Quarters with no observations:
CHL: 0
SST: 0


## 10. Combine and export to CSV

In [10]:
# Combine monthly and quarterly records
combined_df = pd.concat(
    [monthly_df, quarterly_df],
    ignore_index=True,
    sort=False
)

# Sort records
combined_df = combined_df.sort_values(
    ["province_id", "year", "quarter", "period_type", "month"],
    na_position="first"
).reset_index(drop=True)

# Export
combined_df.to_csv(OUTPUT_FILE, index=False)

print("Export complete!")
print("Output:", OUTPUT_FILE)
print("Total records:", len(combined_df))
print("Monthly records:", len(monthly_df))
print("Quarterly records:", len(quarterly_df))

Export complete!
Output: ..\data\processed\nasa\nasa_provincial_quarterly.csv
Total records: 9192
Monthly records: 6888
Quarterly records: 2304
